In [ ]:
from RRAM.Representate import config_ax, setup_paper_plt, config_ax_IV
from statsmodels.distributions.empirical_distribution import ECDF
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Ahora importa el módulo

# Configuración de la figura
setup_paper_plt(plt, latex=True, scaling=2.2)


In [ ]:

ruta_all_data = Path.cwd() / "Datos_Experimentales" / "Medidas_Experimentales_RRAM"
ruta_set = Path.cwd() / "Datos_Experimentales" / "V_Set"

data_path = ruta_set / "V_set_experimental.txt"
results_path = ruta_set / "V_set_experimental.txt"
results_path = ruta_set / "V_set_experimental.txt"

In [ ]:
def data_voltage_extract_sim(data_path: Path, type: str) -> pd.DataFrame:
    # Leer la primera línea para determinar el número de columnas
    with open(data_path, encoding="utf-8") as f:
        first_line = f.readline()
        ncols = len(first_line.strip().split())

    # Determinar nombres según columnas (si suponemos siempre 1 columna de nombre + 2 o 4 voltajes)
    assert ncols in (3, 5), (
        "Formato inesperado: el archivo debe tener 3 o 5 columnas por fila"
    )
    
    if type == "set":
        if ncols == 3:
            dtype = [("Archivo", "U50"), ("V_creacion_1", "f8"), ("V_creacion_2", "f8")]
            col_names = ["Archivo", "V_creacion_1", "V_creacion_2"]
        elif ncols == 5:
            dtype = [("Archivo", "U50"), ("V_creacion_1", "f8"), ("V_creacion_2", "f8"), ("V_creacion_3", "f8"), ("V_creacion_4", "f8")]
            col_names = [ "Archivo", "V_creacion_1", "V_creacion_2", "V_creacion_3", "V_creacion_4"]
    elif type == "reset":
        if ncols == 3:
            dtype = [("Archivo", "U50"), ("V_rotura_1", "f8"), ("V_rotura_2", "f8")]
            col_names = ["Archivo", "V_reset_1", "V_reset_2"]
        elif ncols == 5:
            dtype = [("Archivo", "U50"), ("V_rotura_1", "f8"), ("V_rotura_2", "f8"), ("V_rotura_3", "f8"), ("V_rotura_4", "f8"),]
            col_names = [ "Archivo", "V_rotura_1", "V_rotura_2", "V_rotura_3", "V_rotura_4"]
    
    # Cargar normalmente (sin forzar names explícitamente)
    resultados_txt = np.genfromtxt(
        results_path, dtype=dtype, encoding="utf-8", names=col_names
    )

    # Crear el DataFrame
    df_result_sim = pd.DataFrame(resultados_txt)

    print(df_result_sim)

    #Extraer el número del archivo y añadir como columna
    df_result_sim["Numero"] = (
        df_result_sim["Archivo"]
        .str.extract(r"log_simulacion_(\d+)", expand=False)
        .astype(int)
    )

    return df_result_sim

In [ ]:
def data_voltage_extract_exp(data_path: Path, type: str) -> pd.DataFrame:
    # Leer la primera línea para determinar el número de columnas
    with open(data_path, encoding="utf-8") as f:
        first_line = f.readline()
        ncols = len(first_line.strip().split())


    if type == "set":
        dtype = ([("Archivo", "U50"),("V_set_derivada", "f8"),("V_set_elbow", "f8")])
        col_names = ["Archivo", "V_set_derivada_V", "V_set_elbow_V"]
    elif ncols == 5:
        dtype = [
            ("Archivo", "U50"),
            ("V_creacion_1", "f8"),
            ("V_creacion_2", "f8"),
            ("V_creacion_3", "f8"),
            ("V_creacion_4", "f8"),
        ]
        col_names = [
            "Archivo",
            "V_creacion_1",
            "V_creacion_2",
            "V_creacion_3",
            "V_creacion_4",
        ]

    # Cargar normalmente (sin forzar names explícitamente)
    resultados_txt = np.genfromtxt(
        results_path, dtype=dtype, encoding="utf-8", names=col_names
    )

    # Crear el DataFrame
    df_result_sim = pd.DataFrame(resultados_txt)

    print(df_result_sim)

    # Extraer el número del archivo y añadir como columna
    df_result_sim["Numero"] = (
        df_result_sim["Archivo"]
        .str.extract(r"log_simulacion_(\d+)", expand=False)
        .astype(int)
    )

    return df_result_sim


## nuevo código que extrae directamente desde el metadatos de la simulacion

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# EXTRACCIÓN DE VOLTAJES DE CREACIÓN Y ROTURA POR FILAMENTO
# ─────────────────────────────────────────────────────────────────────────────
import json
from pathlib import Path
import numpy as np

results_dir = Path("Results")

# ── 1. Descubrir simulaciones disponibles ────────────────────────────────────
sim_dirs = sorted(
    [d for d in results_dir.iterdir() if d.is_dir() and d.name.startswith("simulation_")],
    key=lambda d: int(d.name.split("_")[-1]),
)

if not sim_dirs:
    raise FileNotFoundError(f"No se encontraron carpetas de simulación en '{results_dir}'")

# ── 2. Iterar y extraer datos ─────────────────────────────────────────────────
filas: list[dict] = []

for sim_dir in sim_dirs:
    num_sim = int(sim_dir.name.split("_")[-1])
    meta_path = sim_dir / f"sim_metadata_{num_sim}.json"

    if not meta_path.is_file():
        print(f"[WARN] Sim {num_sim}: no existe {meta_path.name}, saltando.")
        continue

    with open(meta_path, "r", encoding="utf-8") as f:
        data: dict = json.load(f)

    num_filamentos: int = data.get("ctes_dict", {}).get("num_filamentos", 0)
    if num_filamentos == 0:
        # Inferir del máximo ID de filamento presente en los dicts
        ids_crea = {v["filamento"] for v in data.get("creaciones_dict", {}).values()}
        ids_rota = {v["filamento"] for v in data.get("roturas_dict", {}).values()}
        num_filamentos = max((ids_crea | ids_rota), default={0}) if (ids_crea | ids_rota) else 0

    # Voltajes de creación: {id_filamento: voltaje}
    v_creacion: dict[int, float] = {}
    for evento in data.get("creaciones_dict", {}).values():
        fid = int(evento["filamento"])
        v_creacion.setdefault(fid, evento["voltaje"])  # primer evento gana

    # Voltajes de rotura: {id_filamento: voltaje}
    v_rotura: dict[int, float] = {}
    for evento in data.get("roturas_dict", {}).values():
        fid = int(evento["filamento"])
        v_rotura.setdefault(fid, evento["voltaje"])

    fila: dict = {"num_simulation": num_sim}
    for fid in range(1, num_filamentos + 1):
        fila[f"v_creacion_CF{fid}"] = v_creacion.get(fid, np.nan)
        fila[f"v_rotura_CF{fid}"] = v_rotura.get(fid, np.nan)

    filas.append(fila)
    print(f"Sim {num_sim:>4d} | creaciones={v_creacion} | roturas={v_rotura}")

# ── 3. Construir cabecera dinámica ────────────────────────────────────────────
if not filas:
    raise RuntimeError("No se extrajeron datos de ninguna simulación.")

# Nº máximo de filamentos encontrado en todo el barrido
max_cf = max(sum(1 for k in fila if k.startswith("v_creacion_CF")) for fila in filas)

header_cols = ["num_simulation"]
for fid in range(1, max_cf + 1):
    header_cols += [f"v_creacion_CF{fid}(V)", f"v_rotura_CF{fid}(V)"]

# ── 4. Ensamblar matriz numérica ──────────────────────────────────────────────
matriz = np.full((len(filas), len(header_cols)), np.nan)

for i, fila in enumerate(filas):
    matriz[i, 0] = fila["num_simulation"]
    for fid in range(1, max_cf + 1):
        col_crea = 1 + (fid - 1) * 2
        col_rota = col_crea + 1
        matriz[i, col_crea] = fila.get(f"v_creacion_CF{fid}", np.nan)
        matriz[i, col_rota] = fila.get(f"v_rotura_CF{fid}", np.nan)

# ── 5. Exportar TXT ───────────────────────────────────────────────────────────
output_path = results_dir / "voltajes_filamentos.txt"

np.savetxt(
    output_path,
    matriz,
    fmt=["%.0f"] + ["%.5f"] * (len(header_cols) - 1),
    delimiter="\t",
    header="\t".join(header_cols),
    comments="",
)

print(f"\nExportado → {output_path}  ({len(filas)} simulaciones, {max_cf} filamentos)")